# Extraction structurée des plans de pistes

Ce notebook transforme les extractions publiques référencées dans `data/imported/` en données exploitables : **pistes avec couleur**, **remontées mécaniques**, **légende**, **activités** et **informations diverses**.

Les résultats publiés se régénèrent sans image ni appel payant. Pour une nouvelle expérience facultative, les plans peuvent être téléchargés depuis leurs URLs sources, puis traités avec un OCR local et une extraction vision structurée. La comparaison avec l'inventaire public des remontées fait ressortir `matched`, `map_only` et `db_only` pour revue humaine.

## 1. Initialisation et paramètres

Laisser `RUN_EXTRACTION = False` pour relire les extractions publiées sans coût. Pour un nouvel essai, télécharger quelques plans, passer la valeur à `True` en gardant `LIMIT = 3`, vérifier les résultats, puis élargir progressivement.

In [ ]:
# À exécuter une seule fois si nécessaire.
# %pip install -e ".[notebook]"
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from PIL import Image
from IPython.display import display


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd / 'resort_slope_maps', cwd.parent):
        if (candidate / 'notebooks').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError(
        'Lancez ce notebook depuis SkiJo, resort_slope_maps ou resort_slope_maps/notebooks.'
    )


PROJECT_ROOT = find_project_root()
PACKAGE_SOURCE = PROJECT_ROOT / 'src'
if str(PACKAGE_SOURCE) not in sys.path:
    sys.path.insert(0, str(PACKAGE_SOURCE))

load_dotenv(PROJECT_ROOT / '.env')

from resort_slope_maps.slope_map_extraction import (
    compare_lifts, export_records, extraction_summary, flatten_records,
    load_published_records, render_plan, run_extractions,
)

IMPORTED_DIR = PROJECT_ROOT / 'data' / 'imported'
INVENTORY_PATH = IMPORTED_DIR / 'slope_map_inventory.csv'
LIFTS_PATH = IMPORTED_DIR / 'database_lifts.csv'
RECORDS_PATH = IMPORTED_DIR / 'extraction_records.jsonl'
required_paths = (INVENTORY_PATH, LIFTS_PATH, RECORDS_PATH)
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Fichiers publics absents : {missing_paths}')

# Sélection des données
SLUGS = []                  # ex. ['chamonix', 'les-karellis']; vide = tous
LIMIT = 3                   # nombre de plans envoyés lors d'une nouvelle extraction

# Extraction
RUN_EXTRACTION = False      # False relit les extractions publiques sans coût
USE_LOCAL_OCR = True        # gratuit; utilisé comme aide, jamais comme vérité unique
FORCE = False               # True ignore le cache et repaie l'extraction
MODEL = os.getenv('OPENAI_MODEL', 'gpt-6-luna')

MAPS_DIR = PROJECT_ROOT / 'data' / 'maps'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs'
CACHE_DIR = PROJECT_ROOT / 'data' / 'cache'
WORKING_DIR = PROJECT_ROOT / 'data' / 'working'
print(f'Modèle : {MODEL} | nouvelle extraction : {RUN_EXTRACTION} | limite : {LIMIT}')


## 2. Inventaire des plans

L'inventaire public contient une ligne par SHA-256. Les fichiers binaires sont facultatifs et restent dans `data/maps/`, hors Git.

In [ ]:
inventory = pd.read_csv(INVENTORY_PATH, parse_dates=['downloaded_at'])
inventory['file_path'] = inventory['asset_name'].map(lambda name: str(MAPS_DIR / name))
inventory['file_exists'] = inventory['file_path'].map(lambda path: Path(path).is_file())
if SLUGS:
    inventory = inventory[inventory['owner_slug'].isin(SLUGS)].copy()
inventory = inventory.sort_values(['source', 'source_key', 'sha256']).reset_index(drop=True)

print(f'{len(inventory):,} plans référencés; {inventory.file_exists.sum():,} fichiers disponibles localement.')
display(
    inventory.groupby(['source', 'suffix'], dropna=False).size()
    .rename('plans').reset_index().sort_values(['source', 'plans'], ascending=[True, False])
)
inventory.head()


## 3. Rattachement au référentiel et chargement des remontées

Les métadonnées publiques associent chaque plan à un domaine ou une station par son slug. Les comparaisons utilisent uniquement le CSV de remontées versionné avec le projet.

In [ ]:
database_lifts = pd.read_csv(LIFTS_PATH)

print(f'{inventory.owner_slug.notna().sum():,}/{len(inventory):,} plans rattachés; '
      f'{len(database_lifts):,} remontées disponibles pour comparaison.')
inventory[['source', 'source_key', 'asset_name', 'owner_kind', 'owner_slug', 'owner_name']].head(20)


## 4. Contrôle visuel d'un fichier préparé

Si un plan a été téléchargé localement, cette cellule montre exactement l'image préparée pour le modèle. Sinon, elle explique comment obtenir un petit échantillon.

In [ ]:
available_maps = inventory[inventory.file_exists]
if available_maps.empty:
    print('Aucun plan local. Utilisez scripts/download_maps.py pour un essai facultatif.')
else:
    preview_row = available_maps.iloc[0]
    preview_pages = render_plan(Path(preview_row.file_path), max_pages=1)
    print(preview_row[['source', 'source_key', 'asset_name', 'owner_slug']].to_dict())
    display(Image.open(__import__('io').BytesIO(preview_pages[0][1])))

## 5. Extraction structurée

Par défaut, le notebook relit les extractions structurées publiées. Une nouvelle extraction facultative utilise le fournisseur OpenAI, un schéma JSON strict et un cache par SHA-256.

In [ ]:
if RUN_EXTRACTION:
    records = await run_extractions(
        inventory, model=MODEL, cache_dir=CACHE_DIR, limit=LIMIT,
        use_local_ocr=USE_LOCAL_OCR, force=FORCE,
    )
else:
    published_records = load_published_records(RECORDS_PATH)
    selected_sha256 = set(inventory['sha256'].astype(str))
    records = [record for record in published_records if record['sha256'] in selected_sha256]
    print('Aucun appel OpenAI. Résultats relus depuis les extractions publiques.')

summary = extraction_summary(records)
display(summary)
if summary.empty:
    print('Aucune extraction publique ne correspond à la sélection.')

## 6. Tables extraites et contrôle qualité

Une confiance basse, des avertissements, une couleur `unknown` ou un nom très court sont des signaux de revue. Le texte OCR brut et les informations de facturation ne font pas partie des données publiques.

In [ ]:
tables = flatten_records(records)
pistes = tables['slopes']
remontees_plan = tables['lifts_on_maps']
legende = tables['legend']
activites = tables['activities']
informations_diverses = tables['other_information']

if not pistes.empty:
    pistes['nom_duplique'] = pistes.duplicated(['plan_sha256', 'name'], keep=False)
    pistes['a_revoir'] = (
        pistes['color'].eq('unknown')
        | pistes['name'].fillna('').str.len().lt(3)
        | pistes['confidence'].fillna(0).lt(0.70)
        | pistes['nom_duplique']
    )
display(pistes.head(30))
display(activites.head(20))

## 7. Comparaison avec les remontées de référence

Le rapprochement enlève accents et abréviations usuelles (`TSD`, `TSF`, `TK`…), puis applique un score de similarité. `map_only` peut signaler une remontée absente de la référence, un plan ancien ou une mauvaise lecture ; `db_only` peut signaler une remontée nouvelle, non dessinée ou mal rattachée.

In [ ]:
lift_comparison = compare_lifts(records, database_lifts, fuzzy_threshold=0.78)
if lift_comparison.empty:
    print('Aucune comparaison disponible pour la sélection.')
else:
    display(lift_comparison.groupby('comparison_status').size().rename('lignes'))
    display(lift_comparison[lift_comparison.a_revoir].head(50))

## 8. Exports

Les CSV analytiques sont régénérés dans `data/outputs/` :

- `slopes.csv` : pistes structurées ;
- `lifts_on_maps.csv` et `lift_comparison.csv` : contrôle des remontées ;
- `legend.csv`, `activities.csv`, `other_information.csv` : enrichissements éditoriaux.

Le JSONL de travail est écrit dans `data/working/`, ignoré par Git comme `data/cache/`. Les fichiers publics de référence restent inchangés dans `data/imported/`.


In [ ]:
if records:
    exported = export_records(records, database_lifts, OUTPUT_DIR, WORKING_DIR)
    for name, path in exported.items():
        print(f'{name:20s} -> {path.relative_to(PROJECT_ROOT)}')
else:
    print('Rien à exporter.')


## 9. Avant une utilisation en production

Le notebook constitue une zone d'analyse, pas encore une source canonique. Avant une intégration : échantillonner plusieurs tailles et styles de plans, mesurer précision/rappel sur les noms et couleurs, résoudre les plans de domaines multi-stations et définir des identifiants stables. Les divergences de remontées doivent rester une file de revue, jamais déclencher une suppression automatique.